# Hof Lecture | Prescriptive Analytics: Transportation Optimization
**Student notebook — runnable BEFORE settings; complete each marked change.**

**Business question:** How many units should each distribution centre ship to each store to meet demand at minimum transport cost?

After the lab, you can identify decision variables, objective and constraints; interpret a valid optimal shipping plan; recognize infeasibility; and explain how capacity and cost changes affect decisions.

Complete **all five experiments in order**, using **BEFORE → CHANGE → AFTER → THINK**. Run each at its starting value, make the marked change, rerun, then answer the questions. Experiments 1–4 independently start from the baseline. Experiment 5 explicitly builds on Experiment 4 by keeping North capacity at 300. No other changes accumulate. No exercise is skipped or assigned an invented time limit.

Only edit the short `# <-- CHANGE HERE` lines. Teacher functions handle the model and reporting. An infeasible scenario is an intended learning outcome, not a notebook error. It has no valid optimal cost or shipment plan.


## Baseline inputs and units
This version follows the **slides**, not the conflicting modified values in the source notebook.

| DC | Available units |
|---|---:|
| DC_East | 600 |
| DC_West | 500 |
| Total | 1,100 |

| Store | Required units | Source |
|---|---:|---|
| Store_A | 250 | Explicit teaching assumption to complete missing slide inputs |
| Store_B | 300 | Explicit teaching assumption to complete missing slide inputs |
| Store_C | 350 | Solution slide; the hands-on slide's “35” is inconsistent |
| Store_D | 200 | Explicit teaching assumption to complete missing slide inputs |
| Total | 1,100 | Slide total |

The slides do not specify A, B and D individually. Their split above is a transparent completion of the missing 750 units, not a recovered fact from the slides. Baseline West=500 follows from the slide's total supply of 1,100 and East=600.

Retain the source notebook's route costs:

| EUR per unit | Store_A | Store_B | Store_C | Store_D |
|---|---:|---:|---:|---:|
| DC_East | 4 | 5 | 7 | 6 |
| DC_West | 6 | 4 | 3 | 5 |

Supply is an upper bound on units shipped, not an obligation to use all capacity. Demand must be met exactly. All quantities refer to the same unspecified planning period and homogeneous product/unit. Costs are **EUR per unit**, and the total objective is **EUR**. Unlike the descriptive notebook, these costs are not EUR per shipment.

For Experiment 4, the slides give North capacity=300 but no route costs. We explicitly assume North unit costs **A=5, B=4, C=4, D=5 EUR/unit**. These are new teaching assumptions; results depend on them. North is optional capacity, with no facility opening or operating cost included.


## What the model chooses
The **decision variable** `x[DC, Store]` is the number of units shipped on that route.

- **Objective:** minimize `sum(unit cost × units shipped)` across all routes.
- **Capacity constraints:** each DC's total shipments must not exceed its available capacity.
- **Demand constraints:** each store must receive exactly its required units.
- **Nonnegativity:** shipped quantities cannot be negative.
- Experiment 5 retains North capacity of 300 and adds `total shipments from DC_West <= 400`. Its BEFORE reference is the North scenario, not the original two-DC baseline.

All listed DC-to-store routes are allowed. Costs are constant per unit. The model is a continuous linear program (LP): quantities may be fractional. This particular transportation structure with integer inputs has an integral optimal solution, but the formulation does not explicitly impose integer variables. No route capacity, transit-time, emissions, stock uncertainty or multi-product interaction is modeled.

**Feasibility first:** With full route connectivity and no additional route constraints, total effective supply below total demand makes this model infeasible. This is different from an expensive but feasible solution.

**No automatic shortages:** This model requires full demand satisfaction. Ordinary unused-capacity slack cannot resolve insufficient supply. Allowing unmet demand would require explicitly introducing shortage variables, defining shortage penalties and changing the service policy; that is a different model and is not silently applied here.

**Compare costs only when both scenarios are optimal.** Signed difference = AFTER − BEFORE, in EUR. If either scenario is infeasible, the cost difference is unavailable, not zero or infinity. Alternative optimal plans can exist; assess feasibility and total cost, not only exact route quantities.

The supplied PuLP formulation is implemented here using SciPy's HiGHS LP solver, with the same objective and constraint structure. This removes the source notebook's runtime PuLP installation step. Both approaches solve the same mathematical model; the baseline inputs have intentionally changed to match the slide totals.


## Teacher setup — run, no edits needed

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
from IPython.display import display

In [ ]:
# TEACHER SETUP — RUN THIS CELL; NO EDITS REQUIRED.

def solve_transport(dcs, stores, costs, west_limit=None):
    """Solve a continuous transport LP, preserving input tables."""
    supply = dcs.set_index("DC")["Supply"].astype(float)
    demand = stores.set_index("Store")["Demand"].astype(float)
    matrix = costs.set_index("DC").loc[supply.index, demand.index].astype(float)
    if not supply.index.is_unique or not demand.index.is_unique:
        raise ValueError("DC and store identifiers must be unique.")
    if (supply < 0).any() or (demand < 0).any():
        raise ValueError("Supply and demand must be nonnegative.")
    if not np.isfinite(matrix.to_numpy()).all():
        raise ValueError("Provide a finite unit cost for every route.")
    nd, ns = len(supply), len(demand)
    c = matrix.to_numpy().ravel()
    a_supply = np.zeros((nd, nd * ns))
    for i in range(nd): a_supply[i, i*ns:(i+1)*ns] = 1
    a_demand = np.zeros((ns, nd * ns))
    for j in range(ns): a_demand[j, j::ns] = 1
    a_ub = a_supply.copy(); b_ub = supply.to_numpy().copy()
    effective = supply.copy()
    if west_limit is not None:
        if west_limit < 0 or "DC_West" not in supply.index:
            raise ValueError("West limit requires DC_West and a nonnegative limit.")
        i = list(supply.index).index("DC_West")
        a_ub = np.vstack([a_ub, a_supply[i]])
        b_ub = np.append(b_ub, float(west_limit))
        effective.loc["DC_West"] = min(supply.loc["DC_West"], west_limit)
    result = linprog(c, A_ub=a_ub, b_ub=b_ub,
                     A_eq=a_demand, b_eq=demand.to_numpy(),
                     bounds=(0, None), method="highs")
    status = {0: "Optimal", 1: "Limit reached", 2: "Infeasible", 3: "Unbounded", 4: "Solver error"}.get(result.status, "Unknown")
    output = {"status": status, "solver_message": result.message,
              "capacity": float(supply.sum()), "effective_capacity": float(effective.sum()),
              "demand": float(demand.sum()),
              "capacity_gap": max(0.0, float(demand.sum()-effective.sum())),
              "total_cost": None, "flows": None, "costs": matrix,
              "supply": supply, "effective_supply": effective, "store_demand": demand}
    if not result.success:
        return output  # No objective or shipment values from an unsuccessful solve.
    flows = pd.DataFrame(result.x.reshape(nd, ns), index=supply.index, columns=demand.index)
    flows[abs(flows) < 1e-8] = 0
    output.update(total_cost=float((flows*matrix).to_numpy().sum()), flows=flows)
    return output

def show_plan(result, title):
    print(title, "| Status:", result["status"])
    display(pd.DataFrame({"Measure": ["Available capacity", "Effective capacity after policy limits", "Required demand", "Aggregate capacity shortfall"],
                          "Units": [result["capacity"], result["effective_capacity"], result["demand"], result["capacity_gap"]]}))
    if result["status"] != "Optimal":
        print("No feasible optimal shipment plan is reported. Total cost is unavailable.")
        print(result["solver_message"])
        return
    print(f"Minimum transport cost: EUR {result['total_cost']:.2f}")
    display(result["flows"].round(3))
    display(pd.DataFrame({"Capacity": result["supply"], "Effective capacity": result["effective_supply"],
                          "Shipped": result["flows"].sum(axis=1),
                          "Unused effective capacity": result["effective_supply"]-result["flows"].sum(axis=1)}).round(3))
    display(pd.DataFrame({"Required": result["store_demand"], "Received": result["flows"].sum(axis=0)}).round(3))
    ax = result["flows"].T.plot.bar(stacked=True, figsize=(7, 3.5))
    ax.set_title(title + " — optimal shipments")
    ax.set_xlabel("Store"); ax.set_ylabel("Units in the planning period")
    ax.tick_params(axis="x", rotation=0)
    ax.figure.tight_layout(); plt.show()

def compare(before, after, title):
    table = pd.DataFrame({"Measure": ["Status", "Effective capacity (units)", "Demand (units)", "Minimum transport cost (EUR)"],
        "BEFORE": [before["status"], before["effective_capacity"], before["demand"], before["total_cost"] if before["total_cost"] is not None else "Unavailable"],
        "AFTER": [after["status"], after["effective_capacity"], after["demand"], after["total_cost"] if after["total_cost"] is not None else "Unavailable"]})
    display(table)
    if before["status"] == after["status"] == "Optimal":
        print(f"Signed cost change (AFTER - BEFORE): EUR {after['total_cost']-before['total_cost']:+.2f}")
    else:
        print("Cost difference unavailable: both scenarios must have valid optimal solutions.")
    show_plan(after, title)

def capacity_scenario(dcs, stores, costs, east_supply):
    x = dcs.copy(deep=True)
    x.loc[x.DC == "DC_East", "Supply"] = east_supply
    return solve_transport(x, stores, costs)

def demand_scenario(dcs, stores, costs, store_c_demand):
    x = stores.copy(deep=True)
    x.loc[x.Store == "Store_C", "Demand"] = store_c_demand
    return solve_transport(dcs, x, costs)

def cost_scenario(dcs, stores, costs, west_extra):
    x = costs.copy(deep=True)
    store_columns = list(stores.Store)
    x.loc[x.DC == "DC_West", store_columns] += west_extra
    return solve_transport(dcs, stores, x)

def north_scenario(dcs, stores, costs, north_capacity, west_limit=None):
    # Explicit additional teaching assumptions: EUR per unit on North routes.
    # Neither the source notebook nor the slides supplies these costs.
    north_costs = {"Store_A": 5.0, "Store_B": 4.0, "Store_C": 4.0, "Store_D": 5.0}
    x = pd.concat([dcs.copy(), pd.DataFrame({"DC": ["DC_North"], "Supply": [north_capacity]})], ignore_index=True)
    y = pd.concat([costs.copy(), pd.DataFrame([{"DC": "DC_North", **north_costs}])], ignore_index=True)
    return solve_transport(x, stores, y, west_limit=west_limit)


## Baseline — inspect BEFORE the experiments
Read capacities, demands and the unit-cost matrix before interpreting the plan.

In [ ]:
# Slide-aligned baseline. Capacities and total demand come from the slides.
# Assumed missing demand split: A=250, B=300, D=200 (sum 750).
baseline_dcs = pd.DataFrame({"DC": ["DC_East", "DC_West"], "Supply": [600, 500]})
baseline_stores = pd.DataFrame({"Store": ["Store_A", "Store_B", "Store_C", "Store_D"],
                                "Demand": [250, 300, 350, 200]})
# Original notebook's route unit costs retained (EUR per unit).
baseline_costs = pd.DataFrame({"DC": ["DC_East", "DC_West"],
    "Store_A": [4., 6.], "Store_B": [5., 4.], "Store_C": [7., 3.], "Store_D": [6., 5.]})

display(baseline_dcs)
display(baseline_stores)
display(baseline_costs)
baseline_result = solve_transport(baseline_dcs, baseline_stores, baseline_costs)
show_plan(baseline_result, "BASELINE")  # <-- OBSERVE OUTPUT

**THINK:** What does the model choose? Which inputs are fixed? How can you verify that every store receives its demand and each DC stays within capacity?

**Your answer:** ...

## Experiment 1 — Capacity reduction
**Business question:** What happens when total capacity no longer covers demand?

**BEFORE:** DC_East capacity = 600.

**CHANGE:** Set EAST_SUPPLY from 600 to 500. Keep West=500 and all demands unchanged. Solve and inspect status before looking for a cost.

**AFTER:** Check status and capacity/demand first. Interpret costs and flows only for an optimal solution.

In [ ]:
EAST_SUPPLY = 600  # <-- CHANGE HERE: 600 -> 500 units
capacity_result = capacity_scenario(baseline_dcs, baseline_stores, baseline_costs, EAST_SUPPLY)
compare(baseline_result, capacity_result, "Experiment 1 — Capacity reduction")  # <-- OBSERVE OUTPUT

**THINK**

1. What are total capacity and demand after the change?
2. Is a valid optimal cost available? Why?
3. What capacity or demand/service-policy change would restore feasibility?

**Your answers:**

1. ...
2. ...
3. ...

## Experiment 2 — Demand spike
**Business question:** Can existing capacity serve an additional 100 units at Store_C?

**BEFORE:** Store_C demand = 350, consistent with the solution slide.

**CHANGE:** Set STORE_C_DEMAND from 350 to 450. Keep all capacities and other store demands fixed. The hands-on slide's starting “35” must be corrected to “350.”

**AFTER:** Check status and capacity/demand first. Interpret costs and flows only for an optimal solution.

In [ ]:
STORE_C_DEMAND = 350  # <-- CHANGE HERE: 350 -> 450 units
demand_result = demand_scenario(baseline_dcs, baseline_stores, baseline_costs, STORE_C_DEMAND)
compare(baseline_result, demand_result, "Experiment 2 — Demand spike")  # <-- OBSERVE OUTPUT

**THINK**

1. How much does total demand rise?
2. Why can the model not simply redistribute shipments to meet all demand?
3. What extra assumption would be required to allow unserved demand?

**Your answers:**

1. ...
2. ...
3. ...

## Experiment 3 — Cost adjustment
**Business question:** Does a uniform West unit-cost increase necessarily change the allocation?

**BEFORE:** No added West route cost.

**CHANGE:** Set WEST_EXTRA_COST from 0 to 2 EUR/unit on every West route. Keep capacities and demands fixed.

**AFTER:** Check status and capacity/demand first. Interpret costs and flows only for an optimal solution.

In [ ]:
WEST_EXTRA_COST = 0  # <-- CHANGE HERE: 0 -> 2 EUR/unit
cost_result = cost_scenario(baseline_dcs, baseline_stores, baseline_costs, WEST_EXTRA_COST)
compare(baseline_result, cost_result, "Experiment 3 — West route costs")  # <-- OBSERVE OUTPUT

**THINK**

1. How many units must West ship in the balanced baseline?
2. What total cost increase follows from adding EUR 2 per unit to that fixed quantity?
3. Must the optimal shipment pattern change? Explain.

**Your answers:**

1. ...
2. ...
3. ...

## Experiment 4 — New DC added
**Business question:** Can optional North capacity lower transport cost under the stated route costs?

**BEFORE:** North capacity = 0; existing East/West network unchanged.

**CHANGE:** Set NORTH_SUPPLY from 0 to 300. North route costs are explicitly assumed A=5, B=4, C=4, D=5 EUR/unit. Keep original demand fixed.

**AFTER:** Check status and capacity/demand first. Interpret costs and flows only for an optimal solution.

In [ ]:
NORTH_SUPPLY = 0  # <-- CHANGE HERE: 0 -> 300 units
north_result = north_scenario(baseline_dcs, baseline_stores, baseline_costs, NORTH_SUPPLY)
compare(baseline_result, north_result, "Experiment 4 — New North DC")  # <-- OBSERVE OUTPUT

**THINK**

1. Does minimum transport cost fall under these North route costs?
2. How much North capacity is used in the displayed optimal plan?
3. Why is transport cost alone insufficient to decide whether to open a real new DC?

**Your answers:**

1. ...
2. ...
3. ...

## Experiment 5 — Combined constraint: North plus West cap
**Business question:** How does a West policy limit change the expanded network plan?

**BEFORE:** Retain North capacity=300. West shipment limit=500, equivalent to its original capacity.

**CHANGE:** Set WEST_SHIPMENT_LIMIT from 500 to 400, retaining East=600, North=300 and unchanged baseline demand. Compare against the North-added network from Experiment 4. The reference is recreated here so the cell can run independently after setup.

**AFTER:** Check status and capacity/demand first. Interpret costs and flows only for an optimal solution.

In [ ]:
WEST_SHIPMENT_LIMIT = 500  # <-- CHANGE HERE: 500 -> 400 units
north_reference = north_scenario(baseline_dcs, baseline_stores, baseline_costs, 300)
west_cap_result = north_scenario(baseline_dcs, baseline_stores, baseline_costs, 300, west_limit=WEST_SHIPMENT_LIMIT)
compare(north_reference, west_cap_result, "Experiment 5 — North plus West cap")  # <-- OBSERVE OUTPUT

**THINK**

1. What is the effective total capacity after the cap?
2. How do cost and allocation change relative to the North-added network?
3. Why would applying the same cap without North make this model infeasible?

**Your answers:**

1. ...
2. ...
3. ...

## Final comparison

| Experiment | What changed? | Solver status | Cost change (EUR) | Business interpretation |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

**Reflection:** Why is checking feasibility the first step before discussing optimization savings? What costs and constraints are missing from this simplified model?

**Your answer:** ...